In [ ]:
## Road Gradient Extraction Workflow

# This notebook estimates road gradient at each collision location by integrating collision coordinates, the OpenStreetMap (OSM) road network, and the Environment Agency LiDAR Composite Digital Terrain Model (DTM). The workflow consists of ten sequential stages. First, the required Python libraries are installed and global parameters (coordinate reference systems, road profile length, LiDAR sampling interval, quality thresholds, and output precision) are defined. The collision dataset is then loaded, validated to ensure latitude and longitude are present, converted into a GeoDataFrame, projected from WGS84 (EPSG:4326) to the British National Grid (EPSG:27700), and used to define the study area.

# The road network covering the study area is downloaded from OpenStreetMap using the **OSMnx** library (via the **OpenStreetMap Overpass API**) and cached locally as a GraphML file for reproducibility and faster subsequent runs. To ensure consistent results, unique coordinate pairs are snapped only once to their nearest road segment using a nearest-neighbour spatial join, after which the snapping results are propagated back to all observations sharing the same location. The exact snapped position and chainage along the road centreline are then calculated, and a local road profile extending 20 m before and 20 m after each snapped location is extracted from the road geometry.

# Ground elevation data are obtained from the **Environment Agency LiDAR Composite Digital Terrain Model (DTM, 1 m resolution)** through the **Defra Data Services Platform Web Coverage Service (WCS) API**. The study area is divided into 5 km × 5 km tiles, each tile is downloaded automatically as a GeoTIFF, and all tiles are combined into a single virtual raster (VRT) using GDAL. Elevation values are sampled every 2 m along each road profile using Rasterio, producing an elevation profile for every collision location.

# Road gradient is calculated as the **endpoint-to-endpoint gradient** over the complete 40 m road profile. Specifically, the elevation difference between the first and last valid LiDAR samples is divided by the corresponding horizontal distance along the road centreline to estimate the average gradient across the profile. The resulting gradient is reported as both slope angle (degrees) and percentage grade. Finally, quality control flags are assigned to identify observations with large snapping distances, insufficient LiDAR samples, bridge or tunnel segments, or implausibly steep gradients. The derived slope variables (`grade_percent`, `slope_deg`, `quality_flag`, `snap_distance_m`, and `n_samples`) are appended directly to the original collision dataset, producing the final output (`collisions_with_slope.csv`) while preserving the original row order.


In [ ]:
# ==========================================================
# Cell 1 - Install required packages
# ==========================================================

!apt-get -qq update
!apt-get -qq install -y gdal-bin libgdal-dev

!pip -q install \
    osmnx \
    geopandas \
    rasterio \
    shapely \
    pyproj \
    scipy \
    networkx \
    pandas \
    numpy \
    matplotlib \
    tqdm \
    scikit-learn \
    fiona

In [ ]:
# ==========================================================
# Cell 2 - Imports and global configuration
# ==========================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import geopandas as gpd
import networkx as nx
import osmnx as ox
import rasterio

from shapely.geometry import Point, LineString
from shapely.ops import substring

from pyproj import Transformer

from scipy.stats import theilslopes

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Progress bars
tqdm.pandas()

# ----------------------------------------------------------
# OSMnx configuration
# ----------------------------------------------------------

ox.settings.use_cache = True
ox.settings.log_console = True

# ----------------------------------------------------------
# Coordinate Reference Systems
# ----------------------------------------------------------

CRS_WGS84 = "EPSG:4326"
CRS_BNG = "EPSG:27700"

# ----------------------------------------------------------
# Road profile parameters
# ----------------------------------------------------------

# metres before point
BACK_DISTANCE = 1

# metres after point
FORWARD_DISTANCE = 20

# spacing between elevation samples
SAMPLE_STEP = 2

# maximum acceptable snap distance
MAX_SNAP_DISTANCE = 20

# minimum profile length required
MIN_PROFILE_LENGTH = 30

# maximum acceptable profile RMSE
MAX_PROFILE_RMSE = 0.35

# buffer around study area when downloading roads
ROAD_BUFFER = 250

NETWORK_TYPE = "drive"

# ----------------------------------------------------------
# Output precision
# ----------------------------------------------------------

ROUND_COORD = 6
ROUND_SLOPE = 3
ROUND_ELEV = 2

print("Configuration loaded.")

In [ ]:
# ==========================================================
# Cell 3 - Prepare point data
# ==========================================================

csv_file = "merged_dataset_2.csv"

print(f"\nReading: {csv_file}")

# ----------------------------------------------------------
# Read CSV
# ----------------------------------------------------------

df = pd.read_csv(csv_file)

# ----------------------------------------------------------
# Required columns
# ----------------------------------------------------------

required = {"latitude", "longitude"}

missing = required - set(df.columns)

if len(missing):
    raise ValueError(
        f"Missing required columns: {missing}\n"
        "CSV must contain latitude and longitude."
    )

# ----------------------------------------------------------
# Create ID if absent
# ----------------------------------------------------------

if "id" not in df.columns:
    df["id"] = np.arange(1, len(df) + 1)

# ----------------------------------------------------------
# Remove invalid coordinates
# ----------------------------------------------------------

df = df[
    df.latitude.between(-90, 90)
    & df.longitude.between(-180, 180)
].copy()

print(f"Valid points: {len(df):,}")

# ----------------------------------------------------------
# Convert to GeoDataFrame
# ----------------------------------------------------------

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df.longitude,
        df.latitude
    ),
    crs=CRS_WGS84
)

# ----------------------------------------------------------
# Project into British National Grid
# ----------------------------------------------------------

gdf = gdf.to_crs(CRS_BNG)

# ----------------------------------------------------------
# Extract projected coordinates
# ----------------------------------------------------------

gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

print("\nProjected coordinate sample:")

display(
    gdf[
        [
            "id",
            "latitude",
            "longitude",
            "x",
            "y"
        ]
    ].head()
)

# ----------------------------------------------------------
# Determine study area
# ----------------------------------------------------------

xmin = gdf.x.min() - ROAD_BUFFER
xmax = gdf.x.max() + ROAD_BUFFER

ymin = gdf.y.min() - ROAD_BUFFER
ymax = gdf.y.max() + ROAD_BUFFER

study_polygon = gpd.GeoSeries(
    [
        Point(xmin, ymin),
        Point(xmin, ymax),
        Point(xmax, ymax),
        Point(xmax, ymin),
    ],
    crs=CRS_BNG,
).unary_union.envelope

study_area = gpd.GeoDataFrame(
    geometry=[study_polygon],
    crs=CRS_BNG,
)

print("\nStudy area created.")

print(f"Bounds (m):")
print(study_area.total_bounds)

# ----------------------------------------------------------
# Quick map
# ----------------------------------------------------------

fig, ax = plt.subplots(figsize=(7,7))

study_area.boundary.plot(ax=ax, color="red")
gdf.plot(ax=ax, markersize=8)

ax.set_title("Input GPS Points")

plt.show()

In [ ]:
# ==========================================================
# Cell 4 - Build or load road network
# ==========================================================

from pathlib import Path

GRAPH_FILE = Path("road_network.graphml")

# ----------------------------------------------------------
# Build study polygon
# ----------------------------------------------------------

study_polygon = (
    gdf.unary_union
       .convex_hull
       .buffer(ROAD_BUFFER)
)

study_polygon = gpd.GeoSeries(
    [study_polygon],
    crs=CRS_BNG
).to_crs(CRS_WGS84).iloc[0]

print("Study polygon prepared.")

# ----------------------------------------------------------
# Load cached graph if available
# ----------------------------------------------------------

if GRAPH_FILE.exists():

    print("Loading cached road network...")

    G = ox.load_graphml(GRAPH_FILE)

else:

    print("Downloading OpenStreetMap roads...")

    G = ox.graph_from_polygon(
        study_polygon,
        network_type=NETWORK_TYPE,
        simplify=True
    )

    print("Projecting graph...")

    G = ox.project_graph(
        G,
        to_crs=CRS_BNG
    )

    ox.save_graphml(
        G,
        GRAPH_FILE
    )

print()

print("Road network ready.")

print(f"Nodes : {len(G.nodes):,}")
print(f"Edges : {len(G.edges):,}")

# ----------------------------------------------------------
# Convert graph to GeoDataFrames
# ----------------------------------------------------------

nodes, edges = ox.graph_to_gdfs(
    G,
    nodes=True,
    edges=True
)

print()

print(f"Road segments: {len(edges):,}")

display(
    edges[
        [
            "highway",
            "name",
            "geometry"
        ]
    ].head()
)

In [ ]:
print(len(gdf))
print(gdf["id"].nunique())

In [ ]:
# ==========================================================
# Cell 5 - Fast snapping (deterministic for duplicate coordinates)
# ==========================================================

print("Preparing edge table...")

edges = edges.copy()

# ----------------------------------------------------------
# Keep original dataset
# ----------------------------------------------------------

gdf_original = gdf.copy()

# ----------------------------------------------------------
# Snap UNIQUE coordinate pairs only
# ----------------------------------------------------------

points = (
    gdf[
        [
            "latitude",
            "longitude",
            "geometry"
        ]
    ]
    .drop_duplicates(
        subset=[
            "latitude",
            "longitude"
        ]
    )
    .copy()
    .reset_index(drop=True)
)

points["point_id"] = np.arange(len(points))

print(f"Unique locations: {len(points):,}")

# ----------------------------------------------------------
# Spatial join
# ----------------------------------------------------------

snapped = gpd.sjoin_nearest(
    points,
    edges.reset_index(),
    how="left",
    distance_col="snap_distance_m"
)

print(f"Matches before deduplication: {len(snapped):,}")

# ----------------------------------------------------------
# Keep ONE nearest edge per location
# ----------------------------------------------------------

snapped = (
    snapped
    .sort_values(
        ["point_id", "snap_distance_m"]
    )
    .drop_duplicates(
        subset="point_id",
        keep="first"
    )
    .reset_index(drop=True)
)

print(f"Matches after deduplication: {len(snapped):,}")

# ----------------------------------------------------------
# Recover road geometry
# ----------------------------------------------------------

def get_edge_geom(row):
    return edges.loc[(row.u, row.v, row.key)].geometry

print("Recovering edge geometries...")

snapped["road_geometry"] = snapped.apply(
    get_edge_geom,
    axis=1
)

# ----------------------------------------------------------
# Chainage
# ----------------------------------------------------------

print("Computing chainage...")

snapped["chainage"] = snapped.apply(
    lambda r: r.road_geometry.project(r.geometry),
    axis=1
)

# ----------------------------------------------------------
# Snapped point
# ----------------------------------------------------------

snapped["snapped_geometry"] = snapped.apply(
    lambda r: r.road_geometry.interpolate(r.chainage),
    axis=1
)

# ----------------------------------------------------------
# Coordinates in BNG
# ----------------------------------------------------------

snapped["snapped_x"] = snapped.snapped_geometry.x
snapped["snapped_y"] = snapped.snapped_geometry.y

# ----------------------------------------------------------
# Convert snapped coordinates back to WGS84
# ----------------------------------------------------------

snapped_ll = gpd.GeoSeries(
    snapped["snapped_geometry"],
    crs=CRS_BNG
).to_crs(CRS_WGS84)

snapped["snapped_longitude"] = snapped_ll.x.values
snapped["snapped_latitude"] = snapped_ll.y.values

# ----------------------------------------------------------
# Rename road attributes
# ----------------------------------------------------------

snapped = snapped.rename(
    columns={
        "name": "road_name",
        "highway": "road_type"
    }
)

# ----------------------------------------------------------
# Merge snapped results back to ALL observations
# ----------------------------------------------------------

lookup = snapped[
    [
        "latitude",
        "longitude",
        "u",
        "v",
        "key",
        "road_geometry",
        "chainage",
        "length",
        "snap_distance_m",
        "snapped_geometry",
        "snapped_x",
        "snapped_y",
        "snapped_longitude",
        "snapped_latitude",
        "road_name",
        "road_type",
        "bridge",
        "tunnel",
    ]
]

gdf = gdf_original.merge(
    lookup,
    on=[
        "latitude",
        "longitude"
    ],
    how="left"
)

print()
print("Finished.")
print(f"Rows after merge: {len(gdf):,}")
print(f"Unique IDs: {gdf['id'].nunique():,}")
print(f"Mean snap distance : {gdf.snap_distance_m.mean():.2f} m")
print(f"Median snap distance : {gdf.snap_distance_m.median():.2f} m")
print(f"Maximum snap distance : {gdf.snap_distance_m.max():.2f} m")

display(
    gdf[
        [
            "id",
            "road_name",
            "road_type",
            "snap_distance_m"
        ]
    ].head()
)

In [ ]:
print("df rows:", len(df))
print("Unique IDs in gdf:", gdf["id"].nunique())
print("Total rows in gdf:", len(gdf))

In [ ]:
# Edge lookup dictionary (much faster than repeated DataFrame lookups)
edge_lookup = {
    idx: row
    for idx, row in edges.iterrows()
}

print(f"Cached {len(edge_lookup):,} road edges.")

In [ ]:
# ==========================================================
# Cell 6 - Build road profile (±20 m on snapped road segment)
# ==========================================================

from shapely.ops import substring
from tqdm.auto import tqdm

tqdm.pandas()

HALF_WINDOW = 20.0      # metres before/after point
PROFILE_STEP = 2.0      # metres (used later)

print("Building road profiles...")


def build_profile(row):

    line = row.road_geometry
    chainage = row.chainage

    if line is None:
        return pd.Series({
            "profile_geometry": None,
            "profile_start_m": np.nan,
            "profile_end_m": np.nan,
            "left_support_m": 0.0,
            "right_support_m": 0.0,
            "support_m": 0.0,
            "short_support": True
        })

    line_length = line.length

    # ----------------------------------------
    # Requested profile limits
    # ----------------------------------------

    start = max(0.0, chainage - HALF_WINDOW)
    end = min(line_length, chainage + HALF_WINDOW)

    # ----------------------------------------
    # Available support
    # ----------------------------------------

    left_support = chainage - start
    right_support = end - chainage

    support = left_support + right_support

    profile = substring(
        line,
        start,
        end
    )

    return pd.Series({

        "profile_geometry": profile,

        "profile_start_m": start,
        "profile_end_m": end,

        "left_support_m": left_support,
        "right_support_m": right_support,

        "support_m": support,

        "short_support": support < (2 * HALF_WINDOW)

    })


profiles = gdf.progress_apply(
    build_profile,
    axis=1
)

gdf = pd.concat(
    [gdf, profiles],
    axis=1
)

print()

print("Finished.")

print()

print(
    "Mean support:",
    round(gdf.support_m.mean(),2),
    "m"
)

print(
    "Median support:",
    round(gdf.support_m.median(),2),
    "m"
)

print(
    "Points with full 40 m support:",
    (gdf.support_m >= 40).sum()
)

print(
    "Points with reduced support:",
    (gdf.support_m < 40).sum()
)

display(

    gdf[
        [
            "id",
            "support_m",
            "left_support_m",
            "right_support_m",
            "short_support"
        ]
    ].head()

)

In [ ]:
print("After Cell 6:", len(gdf))

In [ ]:
# ==========================================================
# Find LiDAR download area
# ==========================================================

import pandas as pd
import geopandas as gpd
from shapely.geometry import box

# change this to your CSV path
CSV_FILE = "/content/merged_dataset_2.csv"

df = pd.read_csv(CSV_FILE)

points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df.longitude,
        df.latitude
    ),
    crs="EPSG:4326"
)

# British National Grid
points_bng = points.to_crs("EPSG:27700")

# add 200 m buffer around all points
xmin, ymin, xmax, ymax = points_bng.total_bounds

aoi = box(
    xmin-200,
    ymin-200,
    xmax+200,
    ymax+200
)

print("BNG bounds:")
print(aoi.bounds)

print("Area km2:")
print(aoi.area / 1e6)

In [ ]:
xmin = 504055.99
ymin = 156548.04
xmax = 559860.02
ymax = 200421.02

In [ ]:
import math
import pandas as pd

# 5 km OS grid
TILE_SIZE = 5000

tiles = []

x0 = math.floor(xmin / TILE_SIZE) * TILE_SIZE
x1 = math.floor(xmax / TILE_SIZE) * TILE_SIZE

y0 = math.floor(ymin / TILE_SIZE) * TILE_SIZE
y1 = math.floor(ymax / TILE_SIZE) * TILE_SIZE


for x in range(int(x0), int(x1)+1, TILE_SIZE):
    for y in range(int(y0), int(y1)+1, TILE_SIZE):
        tiles.append(
            {
                "xmin": x,
                "ymin": y,
                "xmax": x+5000,
                "ymax": y+5000
            }
        )

tiles = pd.DataFrame(tiles)

print("Number of tiles:", len(tiles))

display(tiles.head())

In [ ]:
# ==========================================================
# Create multiple EA upload shapefiles
# each containing ONE polygon
# ==========================================================

import geopandas as gpd
from shapely.geometry import box
from pathlib import Path
import zipfile
import math


xmin = 504055.99
ymin = 156548.04
xmax = 559860.02
ymax = 200421.02


# smaller download size
# try 10 km x 10 km
STEP = 10000


out_base = Path("/content/ea_download_polygons")
out_base.mkdir(exist_ok=True)


count = 0


for x in range(
    math.floor(xmin / STEP) * STEP,
    math.ceil(xmax / STEP) * STEP,
    STEP
):

    for y in range(
        math.floor(ymin / STEP) * STEP,
        math.ceil(ymax / STEP) * STEP,
        STEP
    ):

        poly = box(
            x,
            y,
            x + STEP,
            y + STEP
        )


        tile_gdf = gpd.GeoDataFrame(
            {
                "id":[1]
            },
            geometry=[poly],
            crs="EPSG:27700"
        )


        folder = out_base / f"area_{count}"
        folder.mkdir(exist_ok=True)


        shp = folder / f"area_{count}.shp"


        tile_gdf.to_file(
            shp,
            driver="ESRI Shapefile"
        )


        zipfile_name = (
            f"/content/area_{count}.zip"
        )


        with zipfile.ZipFile(
            zipfile_name,
            "w",
            zipfile.ZIP_DEFLATED
        ) as z:

            for f in folder.glob("*"):
                z.write(
                    f,
                    arcname=f.name
                )


        count += 1


print("Created", count, "upload ZIP files")

In [ ]:
print(len(gdf))
print(gdf["id"].nunique())

In [ ]:
# ==========================================================
# Cell B - Download EA LiDAR DTM tiles (working version)
# ==========================================================

import os
import requests
import pandas as pd
from tqdm.auto import tqdm

OUT_DIR = "/content/lidar_tiles"

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

WCS_URL = (
    "https://environment.data.gov.uk/"
    "spatialdata/lidar-composite-digital-terrain-model-dtm-1m/wcs"
)

COVERAGE_ID = (
    "13787b9a-26a4-4775-8523-806d13af58fc"
    "__Lidar_Composite_Elevation_DTM_1m"
)


def download_tile(row):

    xmin = int(row.xmin)
    ymin = int(row.ymin)
    xmax = int(row.xmax)
    ymax = int(row.ymax)

    filename = f"dtm_{xmin}_{ymin}.tif"

    path = os.path.join(
        OUT_DIR,
        filename
    )

    if os.path.exists(path):
        return "exists"


    params = [
        ("service", "WCS"),
        ("version", "2.0.1"),
        ("request", "GetCoverage"),
        ("coverageId", COVERAGE_ID),
        ("format", "image/tiff"),
        ("subset", f"E({xmin},{xmax})"),
        ("subset", f"N({ymin},{ymax})"),
        (
            "crs",
            "http://www.opengis.net/def/crs/EPSG/0/27700"
        )
    ]


    r = requests.get(
        WCS_URL,
        params=params,
        timeout=300
    )


    if r.status_code != 200:
        return f"HTTP {r.status_code}"


    content_type = r.headers.get(
        "content-type",
        ""
    )


    if "image/tiff" not in content_type:
        return "not_tiff"


    with open(path, "wb") as f:
        f.write(r.content)


    return "ok"


# test one tile first

test_result = download_tile(
    tiles.iloc[0]
)

print("Test result:", test_result)

In [ ]:
# ==========================================================
# Download all LiDAR tiles
# ==========================================================

results = []

for _, row in tqdm(
    tiles.iterrows(),
    total=len(tiles)
):
    results.append(
        download_tile(row)
    )

print(
    pd.Series(results).value_counts()
)

In [ ]:
# ==========================================================
# Cell 7A - Build DTM VRT
# ==========================================================

!apt-get update -qq
!apt-get install -y gdal-bin -qq

!gdalbuildvrt \
/content/london_dtm.vrt \
/content/lidar_tiles/*.tif

In [ ]:
# ==========================================================
# Cell 7B - Check DTM
# ==========================================================

import rasterio

DTM_FILE = "/content/london_dtm.vrt"

with rasterio.open(DTM_FILE) as src:

    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:")
    print(src.bounds)

In [ ]:
# ==========================================================
# Cell 8 - Sample DTM elevations along road profiles
# ==========================================================

import rasterio
import numpy as np
from tqdm.auto import tqdm


DTM_FILE = "/content/london_dtm.vrt"

HALF_WINDOW = 20   # metres before and after point
STEP = 2           # metres between samples


def sample_profile(row, src):

    line = row.road_geometry

    chain = row.chainage

    start = max(
        0,
        chain - HALF_WINDOW
    )

    end = min(
        line.length,
        chain + HALF_WINDOW
    )

    distances = np.arange(
        start,
        end + STEP,
        STEP
    )

    points = [
        line.interpolate(d)
        for d in distances
    ]

    coords = [
        (p.x, p.y)
        for p in points
    ]

    elevations = np.array(
        [
            x[0]
            for x in src.sample(coords)
        ],
        dtype=float
    )


    # remove nodata
    elevations[
        elevations < -100
    ] = np.nan


    valid = ~np.isnan(elevations)


    return {
        "profile_distances": distances[valid],
        "profile_elevations": elevations[valid],
        "n_samples": valid.sum(),
        "profile_start": start,
        "profile_end": end
    }



profiles = []

with rasterio.open(DTM_FILE) as src:

    for _, row in tqdm(
        gdf.iterrows(),
        total=len(gdf)
    ):

        try:

            profiles.append(
                sample_profile(
                    row,
                    src
                )
            )

        except Exception:

            profiles.append(
                {
                    "profile_distances": np.array([]),
                    "profile_elevations": np.array([]),
                    "n_samples": 0,
                    "profile_start": np.nan,
                    "profile_end": np.nan
                }
            )


gdf["profile"] = profiles


gdf["n_samples"] = [
    p["n_samples"]
    for p in profiles
]


print(
    gdf["n_samples"].describe()
)

In [ ]:
import numpy as np
import pandas as pd


def calculate_slope(profile):

    distances = profile["profile_distances"]
    elevations = profile["profile_elevations"]

    if len(distances) < 2:
        return {
            "slope_deg": np.nan,
            "grade_percent": np.nan,
            "signed_grade_percent": np.nan,
            "elevation_change_m": np.nan,
            "quality": "too_few_samples"
        }

    horizontal_distance = distances[-1] - distances[0]

    if horizontal_distance <= 0:
        return {
            "slope_deg": np.nan,
            "grade_percent": np.nan,
            "signed_grade_percent": np.nan,
            "elevation_change_m": np.nan,
            "quality": "invalid_profile"
        }

    elevation_change = elevations[-1] - elevations[0]

    grade = (elevation_change / horizontal_distance) * 100

    slope_deg = np.degrees(np.arctan(grade / 100))

    return {
        "slope_deg": abs(slope_deg),
        "grade_percent": abs(grade),
        "signed_grade_percent": grade,
        "elevation_change_m": elevation_change,
        "quality": "ok"
    }


results = [
    calculate_slope(p)
    for p in gdf.profile
]

slope_df = pd.DataFrame(results)

gdf = pd.concat(
    [
        gdf.reset_index(drop=True),
        slope_df
    ],
    axis=1
)

In [ ]:
# ==========================================================
# Cell 10 - Quality control
# ==========================================================

import numpy as np


gdf["quality_flag"] = "ok"


# Too far from road
gdf.loc[
    gdf.snap_distance_m > 20,
    "quality_flag"
] = "large_snap_distance"


# Too few LiDAR samples
gdf.loc[
    gdf.n_samples < 10,
    "quality_flag"
] = "few_samples"


# Very steep slopes
gdf.loc[
    gdf.grade_percent > 25,
    "quality_flag"
] = "extreme_grade"


# Bridge/tunnel warning
gdf.loc[
    gdf.bridge.notna(),
    "quality_flag"
] = "bridge"


gdf.loc[
    gdf.tunnel.notna(),
    "quality_flag"
] = "tunnel"


print(
    gdf.quality_flag.value_counts()
)

In [ ]:
gdf[
    gdf.grade_percent > 25
][
    [
        "road_name",
        "grade_percent",
        "snap_distance_m",
        "quality_flag"
    ]
]

In [ ]:
gdf[
    gdf.quality_flag=="ok"
][
    "grade_percent"
].describe(
    percentiles=[
        .01,
        .05,
        .25,
        .50,
        .75,
        .95,
        .99
    ]
)

In [ ]:
# ==========================================================
# FINAL MERGE - append slope columns back to original data
# ==========================================================

import pandas as pd
import geopandas as gpd

# Reload the original dataset
original = pd.read_csv("/content/merged_dataset_2.csv")

print("Original rows:", len(original))
print("Slope rows:", len(gdf))

# Safety check
assert len(original) == len(gdf), "Row counts do not match!"

# Copy original collision dataset
final = original.copy()

# Append slope outputs
final["grade_percent"] = gdf["grade_percent"].values
final["slope_deg"] = gdf["slope_deg"].values
final["quality_flag"] = gdf["quality_flag"].values
final["snap_distance_m"] = gdf["snap_distance_m"].values
final["n_samples"] = gdf["n_samples"].values

# Save
output_file = "/content/collisions_with_slope.csv"

final.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Final shape:", final.shape)

In [ ]:
final[[
    "grade_percent",
    "slope_deg",
    "quality_flag",
    "snap_distance_m",
    "n_samples"
]].head()

In [ ]:
import pandas as pd

# Input/output files
input_file = "collisions_with_slope_rounded_renamed.csv"
output_file = "collisions_with_slope_rounded_renamed_2.csv"

# Read CSV
df = pd.read_csv(input_file)

# Move casualty_severity to the end
if "casualty_severity" in df.columns:
    col = df.pop("casualty_severity")
    df["casualty_severity"] = col

# Save
df.to_csv(output_file, index=False)

print(f"Saved to {output_file}")

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/collisions_with_slope.csv")

# Remove column if it exists
df.drop(columns=["slope_deg"], errors="ignore", inplace=True)

# Round numeric columns
cols_to_round = [
   "streetview_distance_m",
   "grade_percent",
   "snap_distance_m",
]

for col in cols_to_round:
    if col in df.columns:
        df[col] = df[col].round(4)

# Move casualty_severity to the end
if "casualty_severity" in df.columns:
    col = df.pop("casualty_severity")
    df["casualty_severity"] = col

# Save
output_file = "/content/collisions_with_slope_final.csv"
df.to_csv(output_file, index=False)

print("Saved:", output_file)
print(df[cols_to_round].head())

In [ ]:
import pandas as pd

# Load final dataset
df = pd.read_csv("/content/collisions_with_slope_final.csv")

# Coordinates with more than one record
dup = df.groupby(["latitude", "longitude"]).filter(lambda x: len(x) > 1)

# Keep only locations where grade differs
different = (
    dup.groupby(["latitude", "longitude"])["grade_percent"]
       .nunique()
       .reset_index(name="n_unique")
)

different = different[different.n_unique > 1]

print("Coordinate pairs with different slopes:", len(different))

if len(different):

    result = (
        dup.merge(
            different[["latitude", "longitude"]],
            on=["latitude", "longitude"]
        )
        .sort_values(["latitude", "longitude"])
    )

    display(
        result[
            [
                "latitude",
                "longitude",
                "grade_percent",
                "snap_distance_m",
                "quality_flag"
            ]
        ]
    )

else:
    print("✓ Every repeated latitude/longitude pair has the same grade_percent.")

In [ ]:
cols = [
    "latitude",
    "longitude",
    "road_name",
    "road_type",
    "u",
    "v",
    "key",
    "chainage",
    "snap_distance_m",
    "grade_percent"
]

display(
    gdf[
        (gdf.latitude == 51.327360) &
        (gdf.longitude == -0.116177)
    ][cols]
)